# Categorical Variables and Binning in pandas

**Python for Data Science II - Healthcare Analytics**

---

## The problem

Almost nothing in a claims file is a number you can do arithmetic with. Diagnoses are codes. Hospitals are labels. Revenue codes *look* like numbers - `301.0`, `636.0` - and pandas will cheerfully compute their mean, which is a completely meaningless quantity.

Encoding is how you turn these into something a model can consume **without lying to it**. Every encoding makes a claim about the data:

- One-hot says *these categories are unrelated*
- Label encoding says *these categories have an order, and equal spacing*
- Manual encoding says *I know something about this domain that the data does not state*

Most encoding errors are not coding errors. They are cases where the method quietly asserted something false and nobody checked.



> **Note on ordering.** Binning comes *before* label encoding on purpose. Label encoding is only honest on an **ordinal** variable, and this dataset contains none out of the box. Part 3 manufactures them, so Part 4 has something legitimate to work on.

---

## 0. Setup and recap

This notebook stands alone, but it starts where the reshaping notebook finished. Recall the key fact about this dataset: **one row is a billed line item, not an admission.** Patient and episode attributes are repeated on every line.

That means we care about two frames at two different **grains**:

- `claims` - one row per **billed line**. `RevenueCode` lives here.
- `episodes` - one row per **admission**. Diagnosis, hospital, age, cost live here.

Encoding at the wrong grain is the single most common mistake in this workflow, and we will demonstrate it explicitly in Part 2.

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 170)

print('pandas', pd.__version__)

pandas 2.3.3


In [2]:
URL = ('https://raw.githubusercontent.com/thousandoaks/Python4DS-I/'
       'main/datasets/HealthcareDataset_PublicRelease.csv')

claims = pd.read_csv(URL, parse_dates=['StartDate', 'EndDate', 'BirthDate'])

# --- derived line-level columns (recap from the reshaping notebook) ---
claims['ServiceCategory'] = claims['RevenueCodeDesc'].str.split(':').str[0].str.strip()
claims['LengthOfStay']    = (claims['EndDate'] - claims['StartDate']).dt.days
claims['AgeAtAdmission']  = (((claims['StartDate'] - claims['BirthDate']).dt.days / 365.25)
                             .round().astype('Int64'))

print(f'claims  (line grain)    : {claims.shape[0]:,} rows x {claims.shape[1]} cols')
claims.head(3)

claims  (line grain)    : 52,563 rows x 20 cols


,Id,MemberName,MemberID,County,MedicalClaim,ClaimItem,HospitalName,HospitalType,StartDate,EndDate,PrincipalDiagnosisDesc,PrincipalDiagnosis,RevenueCodeDesc,RevenueCode,TypeFlag,BirthDate,TotalExpenses,ServiceCategory,LengthOfStay,AgeAtAdmission
0,634363,e659f3f4,6a380a28,6f943458,c1e3436737c77899,18,04b77561,HOSPITAL,2020-01-08,2020-01-08,Epigastric pain,R10.13,DRUGS REQUIRE SPECIFIC ID: DRUGS REQUIRING DET...,636.0,ER,1967-05-13,15.148,DRUGS REQUIRE SPECIFIC ID,0,53
1,634364,e659f3f4,6a380a28,6f943458,c1e3436737c77899,21,04b77561,HOSPITAL,2020-01-08,2020-01-08,Epigastric pain,R10.13,DRUGS REQUIRE SPECIFIC ID: DRUGS REQUIRING DET...,636.0,ER,1967-05-13,3.073,DRUGS REQUIRE SPECIFIC ID,0,53
2,634387,e659f3f4,6a380a28,6f943458,c1e3436737c77899,10,04b77561,HOSPITAL,2020-01-08,2020-01-08,Epigastric pain,R10.13,LABORATORY - CLINICAL DIAGNOSTIC: HEMATOLOGY,305.0,ER,1967-05-13,123.900,LABORATORY - CLINICAL DIAGNOSTIC,0,53


In [3]:
episodes = (claims
            .groupby('MedicalClaim')
            .agg(Member        = ('MemberID', 'first'),
                 Hospital      = ('HospitalName', 'first'),
                 HospitalType  = ('HospitalType', 'first'),
                 County        = ('County', 'first'),
                 EncounterType = ('TypeFlag', 'first'),
                 DxCode        = ('PrincipalDiagnosis', 'first'),
                 DxDesc        = ('PrincipalDiagnosisDesc', 'first'),
                 Admission     = ('StartDate', 'first'),
                 LengthOfStay  = ('LengthOfStay', 'first'),
                 Age           = ('AgeAtAdmission', 'first'),
                 BilledLines   = ('ClaimItem', 'size'),
                 TotalCost     = ('TotalExpenses', 'sum')))

print(f'episodes (claim grain)  : {episodes.shape[0]:,} rows x {episodes.shape[1]} cols')
episodes.head()

episodes (claim grain)  : 3,361 rows x 12 cols


,Member,Hospital,HospitalType,County,EncounterType,DxCode,DxDesc,Admission,LengthOfStay,Age,BilledLines,TotalCost
MedicalClaim,,,,,,,,,,,,
0012a8eb3c2be5f5,2c403f93,ae2f2d9e,HOSPITAL,fd218584,ER,S86.011A,Strain of right Achilles,2020-11-18,0,64,4,4668.692
002fd7d73d8060f1,3eae7881,cf2a3695,HOSPITAL,b021dd12,INP,G50.0,Trigeminal neuralgia,2020-07-17,6,75,24,53501.259
003886fc8ec986d4,7910c083,b592f5ae,HOSPITAL,fd218584,ER,J20.9,Acute bronchitis unspecif,2020-02-26,0,64,18,17115.714
004fa1cd47f65193,9b794cb5,4d103af0,HOSPITAL,02af982d,ER,B02.9,Zoster without complicati,2020-09-06,0,69,9,3672.361
005edafb00d0f6eb,f48d86f4,a9bf1474,HOSPITAL,425a37b2,ER,M79.672,Pain in left foot,2020-03-06,0,73,3,2548.700


---

# Part 1 - Manual encoding

Most tutorials cover this last, as an afterthought. We are doing it **first**, for a practical reason: in healthcare the domain hierarchy is the highest-value transformation available, and applying it *shrinks the problem* the other methods have to solve.

One-hot encoding 150 raw ICD-10 codes gives you 150 sparse columns. Map those codes to their ~15 chapters first and one-hot becomes trivial. The domain knowledge does the heavy lifting; the mechanical encoding just finishes the job.

Manual encoding here means three related things:

1. **Hierarchy extraction** - pull structure out of a code that already contains it
2. **Explicit mapping** - a dictionary plus `.map()`, with a deliberate decision about unmapped values
3. **Rare-category collapse** - fold a long tail into `OTHER`

## 1a. Hierarchy extraction: ICD-10 chapters

ICD-10 codes are not arbitrary strings. `E11.51` is a diabetes code because it starts with `E`; `I49.5` is cardiac because it starts with `I`. The first character encodes the **chapter** - roughly, the body system.

So the naive version is one line:

```python
episodes['Chapter'] = episodes['DxCode'].str[0]
```

**And it is subtly wrong.** The chapter boundaries do not line up perfectly with letters. Neoplasms run `C00-D49`, while blood and immune disorders run `D50-D89`. Both start with `D`. Taking the first character alone merges cancer with anaemia.

This dataset contains `D61.9` (aplastic anaemia), so the bug is live here, not hypothetical. Domain mappings need care - that is precisely why they count as *manual* encoding.

In [4]:
ICD10_CHAPTERS = {
    'A': 'Infectious and parasitic',
    'B': 'Infectious and parasitic',
    'E': 'Endocrine, nutritional, metabolic',
    'F': 'Mental and behavioural',
    'G': 'Nervous system',
    'I': 'Circulatory system',
    'J': 'Respiratory system',
    'K': 'Digestive system',
    'L': 'Skin and subcutaneous',
    'M': 'Musculoskeletal',
    'N': 'Genitourinary system',
    'O': 'Pregnancy and childbirth',
    'P': 'Perinatal conditions',
    'Q': 'Congenital malformations',
    'R': 'Symptoms and abnormal findings',
    'S': 'Injury and poisoning',
    'T': 'Injury and poisoning',
    'V': 'External causes',
    'W': 'External causes',
    'X': 'External causes',
    'Y': 'External causes',
    'Z': 'Health status factors',
}


def icd10_chapter(code):
    """Map an ICD-10-CM code to its chapter, handling the ranges that split a letter."""
    if not isinstance(code, str) or len(code) < 1:
        return 'UNKNOWN'

    letter = code[0].upper()

    # numeric part immediately after the letter, e.g. 'D61.9' -> 61
    digits = ''
    for ch in code[1:]:
        if ch.isdigit():
            digits += ch
        else:
            break
    num = int(digits[:2]) if digits else -1

    # --- the ranges that do NOT respect letter boundaries ---
    if letter == 'C':
        return 'Neoplasms'
    if letter == 'D':
        return 'Neoplasms' if num <= 49 else 'Blood and immune'
    if letter == 'H':
        return 'Eye and adnexa' if num <= 59 else 'Ear and mastoid'

    return ICD10_CHAPTERS.get(letter, 'UNKNOWN')


# quick check on the split cases
for test in ['E11.51', 'I49.5', 'R10.13', 'D61.9', 'C92.11', 'D12.6', 'H25.1', 'H66.9']:
    print(f'{test:<8} -> {icd10_chapter(test)}')

E11.51   -> Endocrine, nutritional, metabolic
I49.5    -> Circulatory system
R10.13   -> Symptoms and abnormal findings
D61.9    -> Blood and immune
C92.11   -> Neoplasms
D12.6    -> Neoplasms
H25.1    -> Eye and adnexa
H66.9    -> Ear and mastoid


In [5]:
episodes['DxChapter'] = episodes['DxCode'].apply(icd10_chapter)

# the naive version, for comparison
episodes['DxLetterOnly'] = episodes['DxCode'].str[0]

print(f'raw ICD-10 codes    : {episodes["DxCode"].nunique():>4} distinct values')
print(f'first letter only   : {episodes["DxLetterOnly"].nunique():>4} distinct values')
print(f'proper chapters     : {episodes["DxChapter"].nunique():>4} distinct values')
print()
episodes['DxChapter'].value_counts().to_frame('admissions')

raw ICD-10 codes    : 1037 distinct values
first letter only   :   21 distinct values
proper chapters     :   20 distinct values



,admissions
DxChapter,
Injury and poisoning,603
Circulatory system,531
Symptoms and abnormal findings,522
Respiratory system,265
Musculoskeletal,263
Digestive system,236
Genitourinary system,190
Infectious and parasitic,161
"Endocrine, nutritional, metabolic",112


### Did the D-split actually matter here?

Check whether any admission would have been misfiled by the naive approach. If the dataset contains both neoplasm and blood codes under `D`, the shortcut would have merged two clinically unrelated groups.

In [6]:
d_codes = episodes.loc[episodes['DxCode'].str.startswith('D', na=False),
                       ['DxCode', 'DxDesc', 'DxChapter']]

if len(d_codes):
    print(f'{len(d_codes)} admissions have a D-code; the naive method puts them all in one bucket.')
    print(f'The correct mapping splits them into: {sorted(d_codes["DxChapter"].unique())}\n')
    display(d_codes.drop_duplicates('DxCode').sort_values('DxCode'))
else:
    print('No D-codes in this extract - but the mapping is still correct for data that has them.')

47 admissions have a D-code; the naive method puts them all in one bucket.
The correct mapping splits them into: ['Blood and immune', 'Neoplasms']



,DxCode,DxDesc,DxChapter
MedicalClaim,,,
efb25bad27a67bd1,D11.0,Benign neoplasm of paroti,Neoplasms
faacab8abcaddaa7,D12.2,Benign neoplasm of ascend,Neoplasms
a9e1c39f7645ba8b,D12.3,Benign neoplasm of transv,Neoplasms
540252cccfd17dda,D25.9,Leiomyoma of uterus unspe,Neoplasms
40e14acb0544cde2,D32.0,Benign neoplasm of cerebr,Neoplasms
6b742d0b7775530c,D32.9,Benign neoplasm of mening,Neoplasms
500260225902ac34,D35.2,Benign neoplasm of pituit,Neoplasms
262b380f32bda82d,D37.4,Neoplasm of uncertain beh,Neoplasms
0f5fd6caebce1365,D49.4,Neoplasm of unspecified b,Neoplasms


## 1b. Explicit mapping with `.map()` - and its silent failure

Revenue codes have block structure too: `25x` is pharmacy, `30x` laboratory, `32x` diagnostic radiology, `36x` operating room, `45x` emergency room. The block is just the code integer-divided by 10.

We will map a **deliberately incomplete** dictionary, because the failure mode is the lesson.

In [7]:
claims['RevBlock'] = (claims['RevenueCode'] // 10).astype('Int64')

# UB-04 revenue code blocks - INTENTIONALLY PARTIAL
REV_BLOCKS = {
    25: 'Pharmacy',
    26: 'IV therapy',
    27: 'Medical/surgical supplies',
    30: 'Laboratory',
    32: 'Radiology - diagnostic',
    35: 'CT scan',
    36: 'Operating room',
    37: 'Anesthesia',
    45: 'Emergency room',
}

claims['RevFamily'] = claims['RevBlock'].map(REV_BLOCKS)

print(f'lines mapped   : {claims["RevFamily"].notna().sum():,}')
print(f'lines UNMAPPED : {claims["RevFamily"].isna().sum():,}  <-- silently became NaN')

lines mapped   : 35,176
lines UNMAPPED : 17,387  <-- silently became NaN


### The silent `NaN`

`.map()` does not warn you. Any key missing from the dictionary becomes `NaN`, and if you do not check, those rows quietly vanish from every subsequent `groupby`, `value_counts`, and plot. Analyses have been published on the strength of a mapping that covered sixty per cent of the data.

**Always audit what failed to map.**

In [8]:
unmapped = (claims.loc[claims['RevFamily'].isna(), ['RevBlock', 'RevenueCodeDesc']]
            .value_counts()
            .rename('lines')
            .reset_index()
            .sort_values('lines', ascending=False))

print('Blocks we forgot, ranked by how much data they represent:\n')
unmapped.head(15)

Blocks we forgot, ranked by how much data they represent:



,RevBlock,RevenueCodeDesc,lines
0,63,DRUGS REQUIRE SPECIFIC ID: DRUGS REQUIRING DET...,3475
1,73,EKG/ECG,2027
2,42,PHYSICAL THERAPY: EVALUATION/RE-EVALUATION,865
3,42,PHYSICAL THERAPY,849
4,41,RESPIRATORY SERVICES,792
5,12,MEDICAL/SURGICAL/GYN,622
6,76,TREATMENT/OBSERVATION ROOM: OBSERVATION ROOM,620
7,20,INTERMEDIATE ICU,559
8,43,OCCUPATIONAL THERAPY,488
9,43,OCCUPATIONAL THERAPY: EVALUATION/RE-EVALUATION,475


In [9]:
# Option 1 - extend the dictionary (best: keeps the domain meaning)
REV_BLOCKS_FULL = {
    **REV_BLOCKS,
    12: 'Room and board',
    20: 'Intensive care',
    22: 'Special charges',
    23: 'Incremental nursing',
    28: 'Oncology',
    29: 'Durable medical equipment',
    31: 'Laboratory',
    33: 'Radiology - therapeutic',
    34: 'Nuclear medicine',
    38: 'Blood',
    39: 'Blood storage and processing',
    40: 'Other imaging',
    41: 'Respiratory services',
    42: 'Physical therapy',
    43: 'Occupational therapy',
    44: 'Speech therapy',
    46: 'Pulmonary function',
    47: 'Audiology',
    48: 'Cardiology',
    49: 'Ambulatory surgical care',
    61: 'MRI',
    63: 'Drugs requiring detail coding',
    71: 'Recovery room',
    73: 'EKG/ECG',
    74: 'EEG',
    92: 'Other diagnostic services',
}

# Option 2 - an explicit, visible fallback for whatever remains
claims['RevFamily'] = (claims['RevBlock']
                       .map(REV_BLOCKS_FULL)
                       .fillna('OTHER - unmapped block'))

still_other = (claims['RevFamily'] == 'OTHER - unmapped block').sum()
print(f'lines now unmapped: {still_other:,} ({still_other / len(claims):.1%})')
print()
claims['RevFamily'].value_counts().to_frame('billed lines').head(15)

lines now unmapped: 1,835 (3.5%)



,billed lines
RevFamily,
Laboratory,16208
Pharmacy,5462
Emergency room,4279
Drugs requiring detail coding,3910
Radiology - diagnostic,2930
Medical/surgical supplies,2847
Physical therapy,2189
CT scan,2171
EKG/ECG,2095


> **The rule.** Never let `.map()` produce `NaN` by accident. Either extend the dictionary or supply an explicit fallback with a name that *announces itself* in a `value_counts()`. A bucket called `OTHER - unmapped block` is honest. A bucket of `NaN` is invisible.

## 1c. Collapsing the rare tail

Even after mapping to chapters, categorical distributions in healthcare have long tails - a handful of chapters cover most admissions, and the rest appear a few times each. Those rare levels are a problem: they become near-empty dummy columns and any statistic computed on three admissions is noise.

The standard fix is to collapse anything below a threshold into `OTHER`. **The threshold is a judgement call the analyst must justify**, which is why this belongs in manual encoding rather than being automated away.

In [10]:
def collapse_rare(s, min_share=0.01, min_count=None, other='OTHER'):
    """Fold categories below a threshold into a single bucket.

    min_share : minimum share of rows a category must hold to survive
    min_count : absolute minimum, overrides min_share when given
    """
    counts = s.value_counts()
    threshold = min_count if min_count is not None else max(1, int(np.ceil(min_share * len(s))))
    keep = counts[counts >= threshold].index
    return s.where(s.isin(keep), other), threshold


chapter_collapsed, thr = collapse_rare(episodes['DxChapter'], min_share=0.01)
episodes['DxChapterGrouped'] = chapter_collapsed

print(f'threshold: a chapter must appear in at least {thr} admissions to survive\n')

comparison = pd.DataFrame({
    'before': episodes['DxChapter'].value_counts(),
    'after' : episodes['DxChapterGrouped'].value_counts(),
})
print(f'levels before : {episodes["DxChapter"].nunique()}')
print(f'levels after  : {episodes["DxChapterGrouped"].nunique()}')
comparison

threshold: a chapter must appear in at least 34 admissions to survive

levels before : 20
levels after  : 16


,before,after
Blood and immune,33.0,NaN
Circulatory system,531.0,531.0
Congenital malformations,2.0,NaN
Digestive system,236.0,236.0
Ear and mastoid,28.0,NaN
"Endocrine, nutritional, metabolic",112.0,112.0
Eye and adnexa,24.0,NaN
Genitourinary system,190.0,190.0
Health status factors,44.0,44.0
Infectious and parasitic,161.0,161.0


### How to choose the threshold

There is no correct answer, but there are defensible ones:

| Basis | Typical rule |
|---|---|
| Statistical | enough rows for the smallest group to support the estimate you intend to make |
| Practical | 1% of rows, or 20-50 absolute, are common defaults |
| Clinical | never collapse a category that matters clinically, however rare |
| Privacy | small cells may be disclosive - many health agencies suppress below 5-11 |

That last row is not optional in real healthcare reporting. A cell containing two patients can identify them.

**Sensitivity check:** if your conclusion changes when the threshold moves from 1% to 2%, the conclusion was resting on the threshold, not on the data.

In [11]:
for share in [0.005, 0.01, 0.02, 0.05]:
    collapsed, t = collapse_rare(episodes['DxChapter'], min_share=share)
    n_other = (collapsed == 'OTHER').sum()
    print(f'min_share={share:<6} threshold={t:<4} levels kept={collapsed.nunique():<3} '
          f'admissions in OTHER={n_other:<5} ({n_other / len(collapsed):.1%})')

min_share=0.005  threshold=17   levels kept=19  admissions in OTHER=3     (0.1%)
min_share=0.01   threshold=34   levels kept=16  admissions in OTHER=88    (2.6%)
min_share=0.02   threshold=68   levels kept=12  admissions in OTHER=289   (8.6%)
min_share=0.05   threshold=169  levels kept=8   admissions in OTHER=751   (22.3%)


---

# Part 2 - One-hot encoding

One column per category, holding 1 if the row belongs to it and 0 otherwise.

The claim it makes: **the categories are unordered and mutually exclusive, and no category is nearer to any other.** For hospitals and diagnosis chapters that is exactly right.

In pandas the tool is `pd.get_dummies`.

In [12]:
dummies_enc = pd.get_dummies(episodes['EncounterType'], prefix='enc', dtype=int)

print('a binary column produces two dummies:\n')
dummies_enc.head()

a binary column produces two dummies:



,enc_ER,enc_INP
MedicalClaim,,
0012a8eb3c2be5f5,1,0
002fd7d73d8060f1,0,1
003886fc8ec986d4,1,0
004fa1cd47f65193,1,0
005edafb00d0f6eb,1,0


In [13]:
print('`dtype=int` matters - without it you get booleans:\n')
print(pd.get_dummies(episodes['EncounterType'], prefix='enc').dtypes.to_string())
print()
print('Booleans are fine for pandas but surprise you later in arithmetic and file exports.')

`dtype=int` matters - without it you get booleans:

enc_ER     bool
enc_INP    bool

Booleans are fine for pandas but surprise you later in arithmetic and file exports.


## 2b. The cardinality explosion

This is why Part 1 came first. One-hot the raw diagnosis code and then the manually-encoded chapter, and compare.

In [14]:
oh_raw     = pd.get_dummies(episodes['DxCode'], prefix='dx', dtype=int)
oh_chapter = pd.get_dummies(episodes['DxChapterGrouped'], prefix='chap', dtype=int)

summary = pd.DataFrame({
    'columns produced': [oh_raw.shape[1], oh_chapter.shape[1]],
    'cells'           : [oh_raw.size, oh_chapter.size],
    'share of 1s'     : [oh_raw.to_numpy().mean(), oh_chapter.to_numpy().mean()],
    'rows per column' : [len(episodes) / oh_raw.shape[1], len(episodes) / oh_chapter.shape[1]],
}, index=['raw ICD-10 code', 'manual chapter encoding'])

summary.style.format({'columns produced': '{:,.0f}', 'cells': '{:,.0f}',
                      'share of 1s': '{:.3%}', 'rows per column': '{:,.1f}'})

,columns produced,cells,share of 1s,rows per column
raw ICD-10 code,"1,037","3,485,357",0.096%,3.2
manual chapter encoding,16,"53,776",6.250%,210.1


The `rows per column` figure is the one to watch. When it drops towards single digits you have columns supported by a handful of observations each - a model cannot learn anything stable from them, and the matrix is almost entirely zeros.

**Rules of thumb for high-cardinality categoricals:**

1. Extract a hierarchy if the code has one (ICD, revenue codes, ATC, postal codes, occupation codes)
2. Collapse the rare tail
3. Only then one-hot
4. If it is still too wide, look at frequency or target encoding - the latter with strict cross-fitting, since it leaks the target if done naively

## 2c. The stability problem

This is the bug most likely to bite you outside a classroom, and it is invisible until it breaks.

`get_dummies` produces columns based on **the values present in the frame you hand it**. Hand it two different frames and you get two different column sets.

In [15]:
rng_split = episodes.sample(frac=0.7, random_state=0)
held_out  = episodes.drop(rng_split.index)

d_train = pd.get_dummies(rng_split['DxChapter'], prefix='chap', dtype=int)
d_test  = pd.get_dummies(held_out['DxChapter'],  prefix='chap', dtype=int)

print(f'columns from the 70% split : {d_train.shape[1]}')
print(f'columns from the 30% split : {d_test.shape[1]}')
print()

only_train = sorted(set(d_train.columns) - set(d_test.columns))
only_test  = sorted(set(d_test.columns) - set(d_train.columns))

print(f'present only in the first  : {only_train}')
print(f'present only in the second : {only_test}')
print()
print('Same column ORDER? ', list(d_train.columns) == list(d_test.columns))

columns from the 70% split : 19
columns from the 30% split : 20

present only in the first  : []
present only in the second : ['chap_Pregnancy and childbirth']

Same column ORDER?  False


If those two lists are not both empty, the frames are incompatible: a model trained on one cannot score the other. And note the last line - even when the *sets* match, the **order** can differ, which silently misaligns columns in any code that indexes by position rather than by name.

### Fix 1 (preferred): fix the categories in the dtype

Cast to `Categorical` with an explicit `categories=` list **before** calling `get_dummies`. The column set is then determined by the dtype, not by the data present. Unseen categories still produce their column - all zeros, which is exactly right.

In [16]:
DX_CATEGORIES = sorted(episodes['DxChapter'].unique())   # decided ONCE, from the reference data

def encode_dx(frame):
    cat = pd.Categorical(frame['DxChapter'], categories=DX_CATEGORIES)
    return pd.get_dummies(cat, prefix='chap', dtype=int)

d_train2 = encode_dx(rng_split)
d_test2  = encode_dx(held_out)

print(f'columns, split A : {d_train2.shape[1]}')
print(f'columns, split B : {d_test2.shape[1]}')
print(f'identical names and order: {list(d_train2.columns) == list(d_test2.columns)}')
print()
print('Columns that are all-zero in split B (category absent, column still present):')
print(' ', [c for c in d_test2.columns if d_test2[c].sum() == 0])

columns, split A : 20
columns, split B : 20
identical names and order: True

Columns that are all-zero in split B (category absent, column still present):
  []


### Fix 2 (repair): reindex against a stored column list

When the encoding has already happened, force the second frame onto the first frame's columns. Missing columns are created and filled with 0; unexpected ones are dropped.

In [17]:
TRAIN_COLUMNS = list(d_train.columns)          # persist this alongside the model

d_test_fixed = d_test.reindex(columns=TRAIN_COLUMNS, fill_value=0)

print(f'before reindex : {d_test.shape[1]} columns')
print(f'after reindex  : {d_test_fixed.shape[1]} columns')
print(f'matches training layout: {list(d_test_fixed.columns) == TRAIN_COLUMNS}')
print()
print('WARNING: a category present in the new data but absent from TRAIN_COLUMNS is')
print('silently discarded. Log it rather than letting it disappear:')
dropped_cats = sorted(set(d_test.columns) - set(TRAIN_COLUMNS))
print(' ', dropped_cats if dropped_cats else 'none in this split')

before reindex : 20 columns
after reindex  : 19 columns
matches training layout: True

silently discarded. Log it rather than letting it disappear:
  ['chap_Pregnancy and childbirth']


## 2d. Encode at the right grain

A callback to the reshaping notebook, and a genuine error that survives code review because the code looks correct.

Suppose we want the share of admissions that are inpatient. One-hot `TypeFlag` and take the mean - but on **which frame**?

In [18]:
by_line    = pd.get_dummies(claims['TypeFlag'], dtype=int).mean()
by_episode = pd.get_dummies(episodes['EncounterType'], dtype=int).mean()

grain = pd.DataFrame({'line grain (WRONG)': by_line,
                      'claim grain (RIGHT)': by_episode})

print('"What share of admissions are inpatient?"\n')
display(grain.style.format('{:.1%}'))

print('\nWhy they differ - inpatient stays generate far more billed lines:')
display(episodes.groupby('EncounterType')['BilledLines']
        .agg(['mean', 'median', 'count'])
        .rename(columns={'mean': 'mean lines per claim',
                         'median': 'median lines',
                         'count': 'admissions'}))

"What share of admissions are inpatient?"



,line grain (WRONG),claim grain (RIGHT)
ER,45.6%,55.7%
INP,54.4%,44.3%



Why they differ - inpatient stays generate far more billed lines:


,mean lines per claim,median lines,admissions
EncounterType,,,
ER,12.815607,10.0,1871
INP,19.184564,19.0,1490


The line-grain answer is not a share of admissions at all - it is a share of *billing lines*, weighted by how much paperwork each admission generated. Both numbers are computed correctly. Only one answers the question asked.

> **Encode at the grain of the question.** Aggregate to the entity you are making claims about, *then* encode.

---

# Part 3 - Binning

Binning (discretisation) converts a continuous variable into an ordered categorical: age becomes an age band, cost becomes a cost tier.

## Why bin at all?

Binning **destroys information**, so it needs a justification. Legitimate ones:

1. **Reporting.** Nobody wants a table with one row per distinct age. Age bands are how clinical results are communicated.
2. **Non-linearity in linear models.** Risk does not rise smoothly with age; it jumps around 65 and again around 85. Bands let a linear model capture a step function it otherwise cannot.
3. **Robustness.** Bins are insensitive to outliers and to the exact value - useful when the measurement is noisy or self-reported.
4. **Small-cell stability.** Grouped estimates are steadier than per-value ones.
5. **Convention and privacy.** Some age bands are mandated by the reporting standard; some are required to prevent re-identification.

## When NOT to bin

- Before a **tree-based model**. Trees find their own splits, and at better cut points than you will guess. Pre-binning strictly discards information.
- When the cut points are **arbitrary**. Bins invented to make a result look good are a well-documented route to a false finding.
- When the variable is **already interpretable** and roughly linear in its effect.

> A specific warning: choosing cut points *after* looking at how they affect your outcome is a form of p-hacking. Fix the bands from domain convention or from the distribution alone, before you look at any relationship.

## 3a. `pd.cut` - bins you choose

`pd.cut` slices on **values you specify**. Use it when the boundaries carry meaning from outside the data.

Age is the clearest case. These are not arbitrary: 18 is legal adulthood, 65 is Medicare eligibility in the US, and 75/85 are conventional cut points for geriatric risk stratification.

In [19]:
AGE_BINS   = [0, 18, 45, 65, 75, 85, 120]
AGE_LABELS = ['0-17', '18-44', '45-64', '65-74', '75-84', '85+']

episodes['AgeBand'] = pd.cut(episodes['Age'].astype('float'),
                             bins=AGE_BINS,
                             labels=AGE_LABELS,
                             right=False)      # [0,18) [18,45) ... left-closed

print('dtype:', episodes['AgeBand'].dtype)
print('ordered:', episodes['AgeBand'].cat.ordered)
print()
episodes['AgeBand'].value_counts().sort_index().to_frame('admissions')

dtype: category
ordered: True



,admissions
AgeBand,
0-17,0
18-44,153
45-64,587
65-74,1203
75-84,1006
85+,412


### Three details students get wrong

**1. `right=` controls which end is closed.** The default `right=True` gives `(0, 18]`, which puts an 18-year-old in the *child* band. With `right=False` you get `[0, 18)` and an 18-year-old is an adult. For age this almost always matters.

**2. Values outside the bins become `NaN`, silently.** If someone is 121 years old, or the age is negative because of a data error, `pd.cut` returns `NaN` rather than complaining. Check.

**3. The result is an ordered `Categorical`.** That is what makes Part 4 possible - the order is stored in the dtype, not implied by the labels.

In [20]:
out_of_range = episodes['AgeBand'].isna() & episodes['Age'].notna()

print(f'ages that fell outside every bin: {out_of_range.sum()}')
if out_of_range.any():
    display(episodes.loc[out_of_range, ['Age', 'Admission', 'DxDesc']].head(10))
else:
    print('none - but never assume this, always check')

print(f'\nmissing ages (NaN in, NaN out): {episodes["Age"].isna().sum()}')

# boundary behaviour, made explicit
demo = pd.DataFrame({'age': [0, 17, 18, 64, 65, 84, 85, 119]})
demo['right=False'] = pd.cut(demo['age'], AGE_BINS, labels=AGE_LABELS, right=False)
demo['right=True']  = pd.cut(demo['age'], AGE_BINS, labels=AGE_LABELS, right=True)
print()
demo

ages that fell outside every bin: 0
none - but never assume this, always check

missing ages (NaN in, NaN out): 0



,age,right=False,right=True
0,0,0-17,NaN
1,17,0-17,0-17
2,18,18-44,0-17
3,64,45-64,45-64
4,65,65-74,45-64
5,84,75-84,75-84
6,85,85+,75-84
7,119,85+,85+


## 3b. `pd.qcut` - bins the data chooses

`qcut` cuts on **quantiles**, so every bin holds roughly the same number of rows. Use it when you have no external basis for boundaries and you want groups of comparable size.

Healthcare cost is the canonical case, because its distribution is brutally skewed - a small number of surgical admissions dwarf everything else. Run `cut` and `qcut` on the same column and the difference is stark.

In [21]:
cost = episodes['TotalCost']

print('How skewed is admission cost?\n')
print(cost.describe(percentiles=[.1, .25, .5, .75, .9, .95, .99]).to_string())
print(f'\nmean / median ratio : {cost.mean() / cost.median():.2f}   (1.0 would be symmetric)')
print(f'top 1% of admissions account for '
      f'{cost.nlargest(max(1, len(cost) // 100)).sum() / cost.sum():.1%} of all spend')

How skewed is admission cost?

count    3.361000e+03
mean     4.277653e+04
std      6.940660e+04
min      1.561000e+02
10%      2.565472e+03
25%      6.211940e+03
50%      2.031355e+04
75%      5.149829e+04
90%      1.059948e+05
95%      1.487698e+05
99%      3.125255e+05
max      1.066310e+06

mean / median ratio : 2.11   (1.0 would be symmetric)
top 1% of admissions account for 11.9% of all spend


In [22]:
equal_width = pd.cut(cost, bins=5)                     # 5 bins of equal COST RANGE
equal_count = pd.qcut(cost, q=5,                       # 5 bins of equal MEMBERSHIP
                      labels=['Q1 lowest', 'Q2', 'Q3', 'Q4', 'Q5 highest'])

side_by_side = pd.DataFrame({
    'pd.cut  (equal width)' : equal_width.value_counts().sort_index().values,
    'pd.qcut (equal count)' : equal_count.value_counts().sort_index().values,
}, index=[f'bin {i + 1}' for i in range(5)])

print('Number of admissions landing in each bin:\n')
display(side_by_side)

print('\nThe cut bins, showing why they are useless here:')
print(equal_width.cat.categories.to_list())

Number of admissions landing in each bin:



,pd.cut (equal width),pd.qcut (equal count)
bin 1,3283,673
bin 2,58,672
bin 3,13,672
bin 4,4,672
bin 5,3,672



The cut bins, showing why they are useless here:
[Interval(-910.054, 213386.816, closed='right'), Interval(213386.816, 426617.531, closed='right'), Interval(426617.531, 639848.247, closed='right'), Interval(639848.247, 853078.962, closed='right'), Interval(853078.962, 1066309.678, closed='right')]


`cut` puts nearly everything in the first bin and leaves the top bins holding a handful of admissions - because it divides the *range*, and the range is set by one extreme claim. Those top bins cannot support any estimate.

`qcut` gives five usable groups. The price is that the boundaries now depend on this particular dataset: refit on new data and the boundaries move, so quintiles are not comparable across cohorts unless you store and reuse the cut points.

In [23]:
episodes['CostTier'] = equal_count

# retrieve the actual boundaries - store these if you need reproducible tiers
_, cost_edges = pd.qcut(cost, q=5, retbins=True)

print('quintile boundaries:')
for i, (lo, hi) in enumerate(zip(cost_edges[:-1], cost_edges[1:]), start=1):
    print(f'  Q{i}: {lo:>12,.0f}  to {hi:>12,.0f}')

print()
episodes.groupby('CostTier', observed=False)['TotalCost'].agg(
    admissions='size', mean_cost='mean', min_cost='min', max_cost='max')

quintile boundaries:
  Q1:          156  to        4,672
  Q2:        4,672  to       13,075
  Q3:       13,075  to       28,314
  Q4:       28,314  to       63,892
  Q5:       63,892  to    1,066,310



,admissions,mean_cost,min_cost,max_cost
CostTier,,,,
Q1 lowest,673,2588.400713,156.100,4672.304
Q2,672,8367.144833,4686.500,13075.300
Q3,672,20406.272083,13105.372,28314.314
Q4,672,43495.114000,28347.025,63892.213
Q5 highest,672,139085.517323,64127.329,1066309.678


## 3c. When `qcut` fails: spiky distributions

`qcut` needs distinct quantile boundaries. If one value dominates the column, several quantiles land on the *same* number, the edges are no longer unique, and pandas raises.

`LengthOfStay` is the trap: emergency visits are same-day, so 0 and 1 account for a large share of all admissions. **Run this and read the error.**

In [24]:
print('LengthOfStay distribution:\n')
print(episodes['LengthOfStay'].value_counts().sort_index().head(10).to_string())
print()

try:
    pd.qcut(episodes['LengthOfStay'], q=5)
except ValueError as err:
    print('qcut raised ValueError:')
    print(' ', err)

LengthOfStay distribution:

LengthOfStay
0    1465
1     455
2     309
3     263
4     216
5     154
6     114
7      95
8      58
9      55

qcut raised ValueError:
  Bin edges must be unique: Index([0.0, 0.0, 0.0, 2.0, 4.0, 129.0], dtype='float64', name='LengthOfStay').
You can drop duplicate edges by setting the 'duplicates' kwarg


In [25]:
# The usual 'fix' - and why it is only half a fix
los_dropped = pd.qcut(episodes['LengthOfStay'], q=5, duplicates='drop')

print(f'requested 5 bins, actually got {los_dropped.cat.categories.size}\n')
print(los_dropped.value_counts().sort_index().to_string())
print('\n`duplicates="drop"` silently gives you fewer bins than you asked for.')
print('Nothing errors. Downstream code expecting 5 tiers now sees fewer.')

requested 5 bins, actually got 3

LengthOfStay
(-0.001, 2.0]    2229
(2.0, 4.0]        479
(4.0, 129.0]      653

`duplicates="drop"` silently gives you fewer bins than you asked for.
Nothing errors. Downstream code expecting 5 tiers now sees fewer.


For a spiky, clinically meaningful variable like length of stay, quantiles are the wrong tool anyway. The natural breaks are **clinical**, not statistical: a same-day visit, a short stay, a week, a long stay. Use `cut` with chosen edges.

In [26]:
LOS_BINS   = [-1, 0, 3, 7, 30, 10_000]
LOS_LABELS = ['Same day', '1-3 days', '4-7 days', '8-30 days', 'Over 30 days']

episodes['LOSBand'] = pd.cut(episodes['LengthOfStay'],
                             bins=LOS_BINS, labels=LOS_LABELS)   # right=True: (-1,0] captures 0

print('Clinically meaningful bands, unequal by design:\n')
display(episodes.groupby('LOSBand', observed=False)
        .agg(admissions=('TotalCost', 'size'),
             mean_cost=('TotalCost', 'mean'),
             pct_inpatient=('EncounterType', lambda s: (s == 'INP').mean())))

print('\nNote the -1 lower edge: with right=True the first interval is (-1, 0],')
print('which is how a zero-day stay gets captured at all.')

Clinically meaningful bands, unequal by design:



,admissions,mean_cost,pct_inpatient
LOSBand,,,
Same day,1465,9054.525659,0.012287
1-3 days,1027,37323.961217,0.598832
4-7 days,579,73145.360523,0.979275
8-30 days,278,155989.929381,1.000000
Over 30 days,12,538246.810500,1.000000



Note the -1 lower edge: with right=True the first interval is (-1, 0],
which is how a zero-day stay gets captured at all.


## 3d. `np.select` - bins from several conditions

`cut` and `qcut` slice one variable. Real classification rules usually combine several, and often are not intervals at all. `np.select` evaluates conditions **in order** and takes the first match, like a chain of if/elif.

In [27]:
age = episodes['Age'].astype('float')
los = episodes['LengthOfStay'].astype('float')

conditions = [
    (age >= 75) & (los >= 8),                    # elderly AND long stay
    (age >= 65) & (los >= 4),
    (age >= 65) | (los >= 8),                    # either factor alone
    (age < 65) & (los <= 1),
]
choices = ['High complexity', 'Elevated', 'Moderate', 'Routine short stay']

episodes['ComplexityBand'] = np.select(conditions, choices, default='Standard')

print('ORDER MATTERS - the first matching condition wins, so put the most')
print('specific rules first. Swap rows 1 and 3 and the High group empties out.\n')

display(episodes.groupby('ComplexityBand')
        .agg(admissions=('TotalCost', 'size'),
             mean_age=('Age', 'mean'),
             mean_los=('LengthOfStay', 'mean'),
             mean_cost=('TotalCost', 'mean'))
        .sort_values('mean_cost', ascending=False))

ORDER MATTERS - the first matching condition wins, so put the most
specific rules first. Swap rows 1 and 3 and the High group empties out.



,admissions,mean_age,mean_los,mean_cost
ComplexityBand,,,,
High complexity,141,81.93617,13.241135,169778.745177
Elevated,622,75.586817,6.922830,96006.881661
Standard,182,54.532967,3.489011,55931.122846
Moderate,1887,76.033916,0.964494,23102.161260
Routine short stay,529,50.852552,0.257089,11991.659694


In [28]:
# Always check the default bucket. A large 'Standard' group means the rules
# do not cover the data and the band is not doing the work you think.
share_default = (episodes['ComplexityBand'] == 'Standard').mean()
print(f'admissions falling through to the default: {share_default:.1%}')
print()
if share_default > 0.25:
    print('That is a lot. The rules need widening, or the default needs a real name.')
else:
    print('Reasonable coverage.')

admissions falling through to the default: 5.4%

Reasonable coverage.


## 3e. The payoff: bins make reporting tables

This is what binning is *for* in healthcare analytics. A continuous variable cannot be a table axis; a band can.

In [29]:
report = episodes.pivot_table(index='AgeBand',
                              columns='EncounterType',
                              values='TotalCost',
                              aggfunc='mean',
                              observed=False)

volume = episodes.pivot_table(index='AgeBand',
                              columns='EncounterType',
                              values='TotalCost',
                              aggfunc='size',
                              observed=False)

print('Mean cost per admission, by age band and encounter type:\n')
display(report.style.format('{:,.0f}', na_rep='-'))

print('\nAdmission counts behind each cell - read these FIRST:\n')
display(volume.fillna(0).astype(int))

Mean cost per admission, by age band and encounter type:



EncounterType,ER,INP
AgeBand,,
18-44,"8,958","40,253"
45-64,"11,147","65,518"
65-74,"11,307","89,397"
75-84,"12,902","87,795"
85+,"13,813","68,384"



Admission counts behind each cell - read these FIRST:



EncounterType,ER,INP
AgeBand,,
0-17,0,0
18-44,115,38
45-64,390,197
65-74,653,550
75-84,501,505
85+,212,200


> **`observed=False`** keeps age bands with zero admissions in the table. With `observed=True` they disappear entirely, which makes a gap in your data look like a category that does not exist. For a *reporting* table you almost always want the empty rows visible. Note that the pandas default for this argument has changed across versions - pass it explicitly rather than relying on it.

> **Read the volume table first.** A cell backed by two admissions is not a finding, and in real reporting it may need suppressing for privacy.

In [30]:
MIN_CELL = 5      # small-cell suppression threshold

suppressed = report.where(volume >= MIN_CELL)

print(f'Same table with cells under {MIN_CELL} admissions suppressed:\n')
suppressed.style.format('{:,.0f}', na_rep='(suppressed)')

Same table with cells under 5 admissions suppressed:



EncounterType,ER,INP
AgeBand,,
18-44,"8,958","40,253"
45-64,"11,147","65,518"
65-74,"11,307","89,397"
75-84,"12,902","87,795"
85+,"13,813","68,384"


---

# Part 4 - Label and ordinal encoding

Label encoding replaces each category with an integer: `0, 1, 2, 3, ...`

It is the most misused encoding in data science, because it is the easiest to apply and its damage is invisible. The integers make a **strong and usually false claim**: that the categories are ordered, and that the gaps between them are equal and meaningful.

In pandas the tool is `.cat.codes` on a `Categorical`.

## 4a. The legitimate case - genuinely ordered data

Part 3 built exactly what this needs. `AgeBand` is an **ordered** `Categorical`: `0-17 < 18-44 < 45-64 < ...` is a true statement, and the order is stored in the dtype rather than assumed.

In [31]:
print('AgeBand is ordered:', episodes['AgeBand'].cat.ordered)
print('category order    :', list(episodes['AgeBand'].cat.categories))
print()

episodes['AgeBandCode'] = episodes['AgeBand'].cat.codes

mapping = pd.DataFrame({
    'band': episodes['AgeBand'].cat.categories,
    'code': range(len(episodes['AgeBand'].cat.categories)),
})
display(mapping)

print('\nThe order is real, so comparisons work:')
print('  number of admissions aged 65+ :',
      int((episodes['AgeBand'] >= '65-74').sum()))

AgeBand is ordered: True
category order    : ['0-17', '18-44', '45-64', '65-74', '75-84', '85+']



,band,code
0,0-17,0
1,18-44,1
2,45-64,2
3,65-74,3
4,75-84,4
5,85+,5



The order is real, so comparisons work:
  number of admissions aged 65+ : 2621


### The `-1` gotcha

`.cat.codes` returns **-1 for missing values**, not `NaN`. That is a plausible-looking integer sitting in a numeric column. It will not raise, it will not show up in `.isna()`, and it will be treated as an ordinary value in every calculation - a category *below* the lowest real band.

In [32]:
n_missing_code = (episodes['AgeBandCode'] == -1).sum()

print(f'rows coded as -1        : {n_missing_code}')
print(f'rows where AgeBand isna : {episodes["AgeBand"].isna().sum()}')
print(f'.isna() on the CODE     : {episodes["AgeBandCode"].isna().sum()}  <-- reports zero regardless')
print()
print('Safe version - restore the missing values explicitly:')

episodes['AgeBandCodeSafe'] = (episodes['AgeBand'].cat.codes
                               .replace(-1, np.nan)
                               .astype('Int64'))

print(f'  missing now visible   : {episodes["AgeBandCodeSafe"].isna().sum()}')

rows coded as -1        : 0
rows where AgeBand isna : 0
.isna() on the CODE     : 0  <-- reports zero regardless

Safe version - restore the missing values explicitly:
  missing now visible   : 0


## 4b. The illegitimate case - and how to prove it is illegitimate

Now label-encode `Hospital`, which is purely nominal.

The demonstration does not need a model. It only needs one observation: **the codes depend on an arbitrary choice.** Encode the same column under two different, equally valid category orderings and the numbers change - so any conclusion that depends on the numbers is a conclusion about the ordering you happened to pick.

In [33]:
hosp_alpha = pd.Categorical(episodes['Hospital'])                       # default: alphabetical
hosp_freq  = pd.Categorical(episodes['Hospital'],
                            categories=episodes['Hospital']
                                       .value_counts().index.tolist())  # by frequency

compare = pd.DataFrame({
    'alphabetical order': pd.Series(hosp_alpha.codes, index=episodes.index),
    'frequency order'   : pd.Series(hosp_freq.codes,  index=episodes.index),
    'hospital'          : episodes['Hospital'].values,
}).drop_duplicates().sort_values('hospital').reset_index(drop=True)

print('The same hospitals, two defensible orderings, two different sets of numbers:\n')
display(compare)

print('\nBoth encodings assert things like:')
print('  "hospital coded 2 is twice hospital coded 1"')
print('  "hospital 1 lies exactly between hospital 0 and hospital 2"')
print('\nNeither statement means anything. The codes are arbitrary, so any result')
print('that changes when you re-order the categories is an artefact of the encoding.')

The same hospitals, two defensible orderings, two different sets of numbers:



,alphabetical order,frequency order,hospital
0,0,111,01b62edc
1,1,88,03840bce
2,2,43,04168b4f
3,3,9,04b77561
4,4,39,04b7f191
...,...,...,...
143,143,98,fc72a0b8
144,144,79,fd89f646
145,145,92,fdc65fe8
146,146,65,ff1c90b6



Both encodings assert things like:
  "hospital coded 2 is twice hospital coded 1"
  "hospital 1 lies exactly between hospital 0 and hospital 2"

Neither statement means anything. The codes are arbitrary, so any result
that changes when you re-order the categories is an artefact of the encoding.


## 4c. So when *is* label encoding acceptable?

| Situation | Verdict |
|---|---|
| Genuinely ordinal feature (`AgeBand`, `CostTier`, severity scales) | **Yes** - the order is real |
| The **target** of a classification model | **Yes** - class labels need to be integers and their order is never used |
| Feeding a **tree-based** model | **Usually acceptable** - a tree can isolate any category through repeated splits, though it needs more splits than one-hot would |
| Nominal feature into a **linear model** | **No** - fabricates order and distance |
| Nominal feature into a **distance-based** method (k-NN, k-means, PCA) | **No** - corrupts the geometry directly |
| Nominal feature you plan to plot on an axis | **No** - the axis implies an order that does not exist |

### A note on `LabelEncoder` vs `OrdinalEncoder`

We have used pandas throughout, but you will meet these in scikit-learn and students conflate them constantly:

- **`LabelEncoder`** is designed for the **target** `y`. It takes 1-D input only.
- **`OrdinalEncoder`** is designed for **features** `X`. It takes 2-D input.

Using `LabelEncoder` on features works column by column, which is why it is so often misused - and it stores no category list, so it cannot handle a category it has not seen. The pandas approach in this notebook, fixing categories in the dtype, is more explicit about exactly that problem.

---

# Part 5 - Summary

## The decision table

| Column looks like | Method | pandas |
|---|---|---|
| Constant | drop | - |
| Identifier (unique per row) | drop, or keep as a key only | - |
| Binary | one dummy | `get_dummies(..., drop_first=True)` |
| Nominal, few levels | one-hot | `get_dummies` |
| Nominal, many levels | hierarchy, then collapse, then one-hot | `.apply(mapper)` -> `collapse_rare` -> `get_dummies` |
| Nominal, numeric-looking code | recognise it, cast to string, then as above | `.astype('string')` |
| Ordinal | ordinal codes | `.cat.codes` on an **ordered** `Categorical` |
| Continuous, external cut points exist | bin with `cut` | `pd.cut(..., bins=[...])` |
| Continuous, skewed, no external basis | bin with `qcut` | `pd.qcut(..., q=n)` |
| Continuous, several interacting rules | `np.select` | `np.select(conds, choices)` |
| Continuous, going into a tree model | **leave it alone** | - |

## Ten mistakes to avoid

1. Encoding a numeric-looking code as a number (`RevenueCode`)
2. One-hot encoding an identifier
3. Label encoding a nominal feature for a linear or distance-based model
4. Letting `.map()` produce `NaN` silently
5. Applying `get_dummies` separately to two frames and assuming the columns match
6. Using `drop_first=True` reflexively, including for tree models
7. Encoding at the wrong grain (line vs claim)
8. Reading `.cat.codes` output without handling the `-1` for missing
9. Using `duplicates='drop'` on `qcut` and not noticing you got fewer bins
10. Choosing bin boundaries after seeing how they affect the result

In [34]:
# Everything applied at once, as a single model-ready frame.

NOMINAL  = ['Hospital', 'EncounterType', 'DxChapterGrouped']
ORDINAL  = ['AgeBand', 'CostTier', 'LOSBand']
NUMERIC  = ['Age', 'LengthOfStay', 'BilledLines', 'TotalCost']

parts = [episodes[NUMERIC]]

for col in NOMINAL:
    cats = sorted(episodes[col].dropna().unique())
    parts.append(pd.get_dummies(pd.Categorical(episodes[col], categories=cats),
                                prefix=col, dtype=int).set_axis(episodes.index))

for col in ORDINAL:
    parts.append(episodes[col].cat.codes.replace(-1, np.nan)
                 .astype('Int64').rename(f'{col}_code'))

model_ready = pd.concat(parts, axis=1)

print(f'episodes    : {episodes.shape[0]:,} rows x {episodes.shape[1]} columns')
print(f'model_ready : {model_ready.shape[0]:,} rows x {model_ready.shape[1]} columns')
print()
model_ready.head()

episodes    : 3,361 rows x 21 columns
model_ready : 3,361 rows x 173 columns



,Age,LengthOfStay,BilledLines,TotalCost,Hospital_01b62edc,Hospital_03840bce,Hospital_04168b4f,Hospital_04b77561,Hospital_04b7f191,Hospital_052ac988,Hospital_0a075ac9,Hospital_1076b523,Hospital_114514f7,Hospital_13dda4ec,Hospital_17b02468,Hospital_1a5406da,Hospital_1aba1b0a,Hospital_1b158493,Hospital_1c8e7f9c,Hospital_1da68531,Hospital_2148dc02,Hospital_21c2e90b,Hospital_226eab93,Hospital_22781f18,Hospital_23ffaec2,...,Hospital_fd89f646,Hospital_fdc65fe8,Hospital_ff1c90b6,Hospital_ffbe51b1,EncounterType_ER,EncounterType_INP,DxChapterGrouped_Circulatory system,DxChapterGrouped_Digestive system,"DxChapterGrouped_Endocrine, nutritional, metabolic",DxChapterGrouped_Genitourinary system,DxChapterGrouped_Health status factors,DxChapterGrouped_Infectious and parasitic,DxChapterGrouped_Injury and poisoning,DxChapterGrouped_Mental and behavioural,DxChapterGrouped_Musculoskeletal,DxChapterGrouped_Neoplasms,DxChapterGrouped_Nervous system,DxChapterGrouped_OTHER,DxChapterGrouped_Respiratory system,DxChapterGrouped_Skin and subcutaneous,DxChapterGrouped_Symptoms and abnormal findings,DxChapterGrouped_UNKNOWN,AgeBand_code,CostTier_code,LOSBand_code
MedicalClaim,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0012a8eb3c2be5f5,64,0,4,4668.692,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,2,0,0
002fd7d73d8060f1,75,6,24,53501.259,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,4,3,2
003886fc8ec986d4,64,0,18,17115.714,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,2,2,0
004fa1cd47f65193,69,0,9,3672.361,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,3,0,0
005edafb00d0f6eb,73,0,3,2548.700,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,3,0,0
